<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Create a Wide-Area Ethernet (Layer 2) Network: User-Defined Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook demonstrates how to create a **wide-area** Layer 2 (Ethernet) network that spans **two different FABRIC sites** using user-defined IP configuration. Unlike a local L2 network (single site), a wide-area L2 network carries Ethernet frames across FABRIC's backbone, enabling direct L2 connectivity between geographically distributed nodes.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create a **wide-area L2 network** that connects nodes at **different FABRIC sites**
2. Select two random FABRIC sites using `get_random_sites()`
3. Use **user-defined configuration** mode to assign specific IP addresses across sites
4. Verify cross-site Layer 2 connectivity using `ping`
5. Understand the difference between local and wide-area L2 networks

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with local L2 networks (see [Local L2 Network - User Defined Config](../create_l2network_basic/create_l2network_basic_config.ipynb))

**Tip:** Wide-area L2 networks take slightly longer to provision than local ones because FABRIC must establish paths across its backbone network.

</div>

## Background: Wide-Area L2 Networks

### Local vs. Wide-Area L2 Networks

A **local** L2 network connects nodes at the same FABRIC site using that site's internal switch. A **wide-area** L2 network extends the Ethernet broadcast domain across FABRIC's optical backbone, connecting nodes at different sites as if they were on the same LAN.


The key difference from a local L2 network is that FABlib automatically detects that the nodes are on different sites and provisions a wide-area VLAN path. You use exactly the same API -- FABlib handles the complexity.

### User-Defined Configuration

As with local L2 networks, you can use `set_mode('config')` to manually assign IP addresses. This ensures predictable addressing regardless of which sites are selected.

### NIC Component Models

| Model | Description | Ports |
|-------|-------------|-------|
| `NIC_Basic` | 100 Gbps Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device | 2 |
| `NIC_ConnectX_6` | 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected by a wide-area L2 Ethernet link (L2STS).

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

Import the FABlib library and the `ipaddress` module for subnet and IP address management.

In [ ]:
# Import Python's ipaddress module for subnet and IP address manipulation
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

# Import the FABlib library and create a manager instance
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select **two different** random FABRIC sites. The `get_random_sites(count=2)` method ensures the sites are distinct, which is required for a wide-area network.

In [ ]:
# Name for the slice -- must be unique among your active slices
slice_name = 'MySlice'

# Select two distinct random FABRIC sites
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node and network names
node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'

## Step 3: Create the Slice with Wide-Area L2 Network

The slice creation process is nearly identical to a local L2 network. The only difference is that each node is placed on a **different site**. FABlib automatically detects this and provisions a wide-area VLAN.

<div class="fab-danger">

**Important:** When using `config` mode, you must call both `set_mode('config')` and `set_ip_addr()` on each interface **before** calling `submit()`. FABlib will apply your IP configuration during post-boot setup.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# Add an L2 network with a user-defined subnet
# FABlib will detect that nodes are on different sites and create a WAN L2 path
net1 = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

# --- Node 1 (on site1) ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set mode to 'config' for user-defined IP assignment
iface1.set_mode('config')
# Connect the interface to our L2 network
net1.add_interface(iface1)
# Assign a specific IP address
iface1.set_ip_addr(IPv4Address("192.168.1.1"))


# --- Node 2 (on site2) ---
node2 = slice.add_node(name=node2_name, site=site2)
# Add a NIC_Basic component and get its first (only) interface
iface2 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set mode to 'config' for user-defined IP assignment
iface2.set_mode('config')
# Connect the interface to our L2 network
net1.add_interface(iface2)
# Assign a specific IP address
iface2.set_ip_addr(IPv4Address("192.168.1.2"))


# Submit the slice request to FABRIC
# Wide-area slices may take slightly longer than local ones (~3-7 minutes)
slice.submit()

<div class="fab-success">

**What just happened?** FABRIC provisioned VMs at two geographically separate sites, established a VLAN path across its backbone network, and configured both interfaces with your specified IP addresses. The two nodes can now communicate at Layer 2 as if they were on the same local switch.

</div>

## Step 4: Run the Experiment

We verify cross-site connectivity by pinging Node2 from Node1. Since this is a wide-area network, you should observe higher latency than a local L2 network due to the physical distance between sites.

<div class="fab-warning">

**Tip:** The round-trip time in the ping output reflects the real geographic distance between the two FABRIC sites. Compare this with a local L2 network ping to appreciate the difference!

</div>

In [ ]:
# Retrieve the slice (useful if reconnecting in a new session)
slice = fablib.get_slice(slice_name)

# Get references to both nodes
node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

# Look up Node2's IP address on the network (should be 192.168.1.2)
node2_addr = node2.get_interface(network_name=network_name).get_ip_addr()

# Ping Node2 from Node1 to verify wide-area L2 connectivity
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `ping` fails between nodes | Interface not configured | Verify both interfaces have `set_mode('config')` and `set_ip_addr()` called before `submit()` |
| `ping` shows 100% packet loss | WAN path not established | Wait a few minutes after slice becomes active; WAN paths may take extra time to converge |
| Both nodes placed on same site | `get_random_sites()` returned duplicates | This should not happen with `count=2`; verify by checking `site1 != site2` |
| Slice stuck in `Configuring` | One site may be down | Retry with different sites or specify sites manually |
| `No resources available` error | Sites lack NIC_Basic capacity | Use `fablib.list_sites()` to find sites with available SmartNIC VFs |
| Higher latency than expected | Physical distance between sites | This is expected behavior -- WAN L2 latency depends on geography |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_sites(count)` | Select multiple distinct random FABRIC sites | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_l2network(name, subnet)` | Add a Layer 2 network to the slice | [add_l2network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l2network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `node.add_component(model, name)` | Add a NIC or other component to a node | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `component.get_interfaces()` | Get the list of interfaces on a component | [get_interfaces](https://fabric-fablib.readthedocs.io/en/latest/component.html#fabrictestbed_extensions.fablib.component.Component.get_interfaces) |
| `iface.set_mode(mode)` | Set interface configuration mode (`auto` or `config`) | [set_mode](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_mode) |
| `iface.set_ip_addr(addr)` | Assign an IP address to the interface | [set_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_ip_addr) |
| `net.add_interface(iface)` | Connect an interface to a network | [add_interface](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.add_interface) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you can create wide-area L2 networks with user-defined configuration, explore these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **L2 with Explicit Routes** | [create_l2network_wide_area_ero_auto](./create_l2network_wide_area_ero_auto.ipynb) | Control network paths and bandwidth with Explicit Route Options |
| **Local L2 Network** | [create_l2network_basic_config](../create_l2network_basic/create_l2network_basic_config.ipynb) | Create a single-site L2 network with user-defined config |
| **FABnet IPv4 (L3)** | [create_l3network_fabnet_ipv4_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Use FABRIC's managed Layer 3 networking for multi-site experiments |
| **DHCP on L2 Networks** | [dhcp_l2_network](../dhcp_l2_network/dhcp.ipynb) | Set up a DHCP server to dynamically assign IPs on L2 |
| **Storage Benchmarking** | [benchmarking_storage](../benchmarking_storage/benchmarking_storage.ipynb) | Benchmark local disk and NVMe storage on FABRIC |